# Grand Case Study 2: Image Processing & Computer Vision

Welcome to the second Grand Case Study. Linear Algebra is the absolute mathematical language of Computer Vision. In this notebook, we will use core linear algebra concepts to understand how computers "see", compress, and alter images.

We will ask practical computer vision questions and use math to answer them.

## Question 1: How does a computer actually store an image?
**The Problem:** We need a way to mathematically represent a photograph.

**The Linear Algebra Answer: Matrices and Tensors**
A computer doesn't see a landscape; it sees a 2D Matrix of numbers ranging from 0 (black) to 255 (white). For a color image, it is a 3D Tensor (Width $\times$ Height $\times$ 3 Color Channels: Red, Green, Blue).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.color import rgb2gray

# Load a sample image of an astronaut
img_color = data.astronaut()
img_gray = rgb2gray(img_color)

print("Color Image Shape (3D Tensor):", img_color.shape)
print("Grayscale Image Shape (2D Matrix):", img_gray.shape)

plt.figure(figsize=(8,4))
plt.subplot(1, 2, 1)
plt.title("The Visual Image")
plt.imshow(img_gray, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Linear Algebra view (Top Left 5x5)")
# Printing the top-left 5x5 sub-matrix
plt.text(0.1, 0.4, str(np.round(img_gray[0:5, 0:5], 2)), fontsize=10)
plt.axis('off')
plt.show()

## Question 2: How can we mathematically manipulate an image (e.g., extracting its edges)?
**The Problem:** In order for a self-driving car to stay on the road, it needs an algorithm to instantly highlight lines and edges and ignore flat asphalt.

**The Linear Algebra Answer: Linear Transformations via Convolution**
We can slide a small mathematical matrix (called a kernel or filter) over the larger image matrix. By taking the inner product (dot product) between this small filter matrix and chunks of the image, we apply a Linear Transformation that highlights vertical or horizontal edges.

In [ ]:
from scipy.signal import convolve2d

# Sobel Filter (a special transformation matrix designed to find vertical edges)
sobel_vertical = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]])

# Apply the linear transformation to the image
edges = convolve2d(img_gray, sobel_vertical, mode='same', boundary='symm')

plt.figure(figsize=(6,6))
plt.imshow(np.abs(edges), cmap='gray')
plt.title("Vertical Edges Extracted via Matrix Multiplication")
plt.axis('off')
plt.show()
# Notice how vertical features (like the side of her helmet) shine bright white!

## Question 3: Self-driving cars need to process millions of HD images. How can we compress images massively while preserving visual structure?
**The Problem:** Transmitting raw 4K video feeds is impossible over cellular networks. We need a way to store 90% of the image's "essence" using only 10% of the memory.

**The Linear Algebra Answer: Singular Value Decomposition (SVD)**
Because natural images have a lot of redundant structure (groups of pixels that look the same), the image matrix is technically 'Low Rank'. We can apply SVD ($A = U \Sigma V^T$) to factorize the image, and then throw away almost all of the small singular values, saving immense amounts of data.

In [ ]:
# Perform SVD on the Grayscale Matrix
U, S, VT = np.linalg.svd(img_gray, full_matrices=False)

# 512 singular values exist. Let's ONLY keep the top 20 (Compressing roughly by 96%!) 
k = 20
U_reduced = U[:, :k]
S_reduced = np.diag(S[:k])
VT_reduced = VT[:k, :]

# Mathematical reconstruction using only the top 20 singular values
img_compressed = U_reduced.dot(S_reduced).dot(VT_reduced)

plt.figure(figsize=(8,4))
plt.subplot(1, 2, 1)
plt.title("Original (k=512)")
plt.imshow(img_gray, cmap='gray'); plt.axis('off')

plt.subplot(1, 2, 2)
plt.title(f"SVD Compressed (k={k})")
plt.imshow(img_compressed, cmap='gray'); plt.axis('off')
plt.show()
# The image is slightly blurry, but the astronaut structure is entirely preserved using just a fraction of the data!

## Question 4: How does early Facial Recognition actually work mathematically?
**The Problem:** If I have a database of 100 face images, how do I compare a new security camera image to my database rapidly?

**The Linear Algebra Answer: Eigenvalues, Eigenvectors, and "Eigenfaces"**
Instead of trying to match a face pixel-by-pixel, we compute the Covariance Matrix of our *entire* face database. The **Eigenvectors** of this massive matrix represent the "Ghosts" of human variance: one eigenvector might capture "Lighting direction", another captures "Big eyes vs Small eyes". These are called **Eigenfaces**.

When a new face arrives, we map it into this low-dimensional "Eigenface space". We then simply calculate the distance (L2 Norm) between the new face and the database faces in this abstract space. If the distance is nearly zero, we have a match!